In [1]:
# vae_training_notebook.ipynb
%load_ext autoreload
%autoreload 2
# Cell 1: 导入和 GPU 检查
import os
# 在任何 torch 相关操作前设置环境变量
os.environ["PYTORCH_DISABLE_DYNAMO"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"  # 额外禁用 CUDA DSA
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# 现在导入 torch
import torch

# 额外禁用 torch 编译功能
torch._dynamo.config.disable = True
torch._dynamo.config.suppress_errors = True

# 是否有可用 GPU
print("CUDA available:", torch.cuda.is_available())

# 当前 GPU 名称
if torch.cuda.is_available():
    print("Current GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



CUDA available: True
Current GPU: NVIDIA GeForce RTX 3070 Laptop GPU


In [2]:
# Cell 2: VAE 编码器引用（已根据文件结构修正）

# ========================================================
# 方案 A：使用带缩放逻辑的版本 (当前激活)
# 对应文件: DeepLearning/Encoders/VAE/VAE_Encoder_scaled.py
# ========================================================
from DeepLearning.Encoders.VAE.VAE_Encoder_scaled import CustomVAEEncoder

print(f"Active: Loaded Scaled VAE Encoder (log1p + max_norm).")


# ========================================================
# 方案 B：使用无缩放逻辑的版本 (已注释，用于对比测试)
# 对应文件: DeepLearning/Encoders/VAE/VAE_Encoder.py
# ========================================================
"""
# 如果需要回滚到无缩放版本，请注释掉上面的 A 部分，取消下面两行的注释：
# from DeepLearning.Encoders.VAE.VAE_Encoder import CustomVAEEncoder
# print("Active: Loaded Unscaled VAE Encoder (Legacy).")
"""

ImportError: DLL load failed while importing _multiarray_umath: 找不到指定的模块。

ImportError: DLL load failed while importing _multiarray_umath: 找不到指定的模块。

Loaded VAE encoder with fixed obs_max: 7.288927694521257
Active: Loaded Scaled VAE Encoder (log1p + max_norm).


'\n# 如果需要回滚到无缩放版本，请注释掉上面的 A 部分，取消下面两行的注释：\n# from DeepLearning.Encoders.VAE.VAE_Encoder import CustomVAEEncoder\n# print("Active: Loaded Unscaled VAE Encoder (Legacy).")\n'

In [4]:
# Cell 2.5: 初始阶段 - 与randomagent对战（SelfPlayDense 环境）
from DeepLearning.Environments.Thesis.SelfPlay import SelfPlayDense
from DeepLearning.Encoders.MainGame.ActionMask.GetActionMask import getActionMask
from DeepLearning.Encoders.MainGame.Observation.get_observation_full import getObservationFull
from DeepLearning.CustomMaskablePPO import MaskablePPO
import os

# 配置 VAE 维度（可调整）
vae_dim = 64

# 环境初始化（SelfPlayDense，3个RandomAgent对手）
# trading=False: 禁用交易
# selfPlay=False: 使用 RandomAgent 对手
env = SelfPlayDense(trading=False, selfPlay=False)
env.denseRewards = True  # 启用 dense rewards

# 设置 Reward 机制
env.winReward = True
env.winRewardAmount = 50  # Win: +50
# Lose: -5 * (10 - VP)，在 endGame 中自动处理

actionMask = getActionMask
observation = getObservationFull

# PPO 配置
os.environ["UPDATE_MODELS_DIST"] = "False"
netArchDict = dict(pi=[128, 128, 128], vf=[128, 128, 128])
gamma = 0.99
n_steps = 2048

#saveName_init = f"ZKA_VAE_RandomOpponents_{vae_dim}d_5M"

# 保存Hengyizhang 的版本，用于对比新的vae结果
saveName_init = f"ZKA_scaledVAE_RandomOpponents_{vae_dim}d_5M"

savePath_init = f"DeepLearning/Models/ZKA_model/{saveName_init}"

# 创建 PPO 模型，集成 VAE 编码器
model = MaskablePPO(
    "MlpPolicy",
    env,
    verbose=1,
    device=device,
    policy_kwargs=dict(
        features_extractor_class=CustomVAEEncoder,
        features_extractor_kwargs=dict(features_dim=vae_dim),
        net_arch=netArchDict
    ),
    gamma=gamma,
    n_steps=n_steps,
    getActionMask=actionMask,
    getObservation=observation,
    savePath=savePath_init,
    tensorboard_log="./tensorboard_logs_thesis/"
)

model.savePath = savePath_init
print("Policy device:", next(model.policy.parameters()).device)
print("Reward settings: Win=+50, Lose=-5*(10-VP)")
print("Trading disabled: trading=False")
print("SelfPlay disabled: selfPlay=False (using RandomAgent opponents)")

# 训练 5M timesteps
model.learn(total_timesteps=5_000_000, tb_log_name=saveName_init, reset_num_timesteps=False)

# 保存初始阶段模型
model.save(savePath_init)
print(f"Initial stage model saved to: {savePath_init}")


C:\Users\ZKA\AppData\Local\Temp\ipykernel_17344\1232386592.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
C:\Users\ZKA\AppData\Local\Temp\ipykernel_17344\1232386592.py:2

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Policy device: cuda:0
Reward settings: Win=+50, Lose=-5*(10-VP)
Trading disabled: trading=False
SelfPlay disabled: selfPlay=False (using RandomAgent opponents)
Logging to ./tensorboard_logs_thesis/ZKA_scaledVAE_RandomOpponents_64d_5M_0
CheckingWinRate(Distribution): 0.1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 92.3     |
|    ep_rew_mean     | 33.2     |
| time/              |          |
|    fps             | 243      |
|    iterations      | 1        |
|    time_elapsed    | 8        |
|    total_timesteps | 2048     |
---------------------------------
CheckingWinRate(Distribution): 0.19
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 90.8        |
|    ep_rew_mean          | 41.3        |
| time/                   |             |
|    fps                  | 213         |
|    i

In [7]:

# Cell 3: 阶段1 - 禁用交易的selfplay
from DeepLearning.Environments.Thesis.SelfPlay import SelfPlayDense
from DeepLearning.Encoders.MainGame.ActionMask.GetActionMask import getActionMask
from DeepLearning.Encoders.MainGame.Observation.get_observation_full import getObservationFull
from DeepLearning.CustomMaskablePPO import MaskablePPO
import os

# 配置 VAE 维度（可调整）
vae_dim = 64

# PPO 配置
os.environ["UPDATE_MODELS_DIST"] = "True"
netArchDict = dict(pi=[128, 128, 128], vf=[128, 128, 128])
gamma = 0.99
n_steps = 2048

# 加载初始阶段模型
stage0_model_path = savePath_init + ".zip"  # 假设保存为 zip
print("Load model from:", stage0_model_path)
model = MaskablePPO.load(stage0_model_path, device=device)

# 确保 VAE 仍然冻结
for param in model.policy.features_extractor.parameters():
    param.requires_grad = False

# 切换到新的环境（禁用交易，selfPlay=True）
env = SelfPlayDense(trading=False, selfPlay=True)
env.denseRewards = True
env.bankTradeReward = True

# 更新模型的环境
model.set_env(env)

# 配置保存路径
saveName = f"ZKA_VAE_NoTrade_{vae_dim}d_5M"
savePath = f"DeepLearning/Models/ZKA_model/{saveName}"
model.savePath = savePath

print("Policy device:", next(model.policy.parameters()).device)
print("Loaded model from initial stage, continuing training...")

# 训练 5M timesteps
model.learn(total_timesteps=5_000_000, tb_log_name=saveName, reset_num_timesteps=False)

# 保存阶段1模型
model.save(savePath)
print(f"Stage 1 model saved to: {savePath}")


Load model from: DeepLearning/Models/ZKA_model/ZKA_VAE_RandomOpponents_64d_5M.zip


C:\Users\ZKA\AppData\Local\Temp\ipykernel_17344\1232386592.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
C:\Users\ZKA\AppData\Local\Temp\ipykernel_17344\1232386592.py:2

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Policy device: cuda:0
Loaded model from initial stage, continuing training...
Logging to ./tensorboard_logs_thesis/ZKA_VAE_NoTrade_64d_5M_0


C:\Users\ZKA\AppData\Local\Temp\ipykernel_17344\1232386592.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


CheckingWinRate(Distribution): 0.12
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 58.6     |
|    ep_rew_mean     | 93       |
| time/              |          |
|    fps             | 89       |
|    iterations      | 1        |
|    time_elapsed    | 23       |
|    total_timesteps | 5003264  |
---------------------------------
CheckingWinRate(Distribution): 0.21
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 57          |
|    ep_rew_mean          | 64.3        |
| time/                   |             |
|    fps                  | 82          |
|    iterations           | 2           |
|    time_elapsed         | 49          |
|    total_timesteps      | 5005312     |
| train/                  |             |
|    approx_kl            | 0.011119346 |
|    clip_fraction        | 0.0933      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.755      |


KeyboardInterrupt: 

In [ ]:
model.learn(total_timesteps=1_000_000, tb_log_name=saveName, reset_num_timesteps=False)

# 保存阶段1模型
model.save(savePath)
print(f"Stage 1 model saved to: {savePath}")


In [ ]:

# Cell 4: 阶段2 - 启用完整功能的高级学习
from Client.DeepLearning.Environments.SelfPlayTrading import SelfPlayTrading

# 加载阶段1模型
stage1_model_path = savePath + ".zip"  # 假设保存为 zip
model = MaskablePPO.load(stage1_model_path, device=device)

# 切换到启用交易的环境
env = SelfPlayTrading(trading=True)

# 确保 VAE 仍然冻结
for param in model.policy.features_extractor.parameters():
    param.requires_grad = False

saveName_stage2 = f"ZKA_VAE_WithTrade_{vae_dim}d_6M"
savePath_stage2 = f"DeepLearning/Models/ZKA_model/{saveName_stage2}"

model.savePath = savePath_stage2
print("Policy device:", next(model.policy.parameters()).device)

# 继续训练 6M timesteps
model.learn(total_timesteps=6_000_000, tb_log_name=saveName_stage2, reset_num_timesteps=False)

# 保存最终模型
model.save(savePath_stage2)
print(f"Final model saved to: {savePath_stage2}")
